# Segmentación por **K-means** con validación cruzada de **K = 5** y test fijo

Aplica **K-means** a todas las imágenes de
`DATASET_FINAL2.mat`.

* validación cruzada **estratificada por tipo de lesión**, K = 5 pliegues
* las imágenes de índice **23–27** (con imagen registrada) son **siempre test** y nuncaentran en entrenamiento ni en validación
* el resto de imágenes rota: en cada pliegue, 80 % train+val (reparto interno 80/20) y
  20 % test, que se suma al test fijo
* métricas reportadas como **media** entre pliegues, desglosadas en test total,
  solo rotatorias y solo fijas


### 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 2. Parámetros

In [ ]:
RUTA_MAT       = "/C:/Users/josem/Desktop/DATASET_FINAL2.pdf"

UMBRAL_DICE   = 0.9      # solo se muestran las imágenes que superen este Dice
MAX_MOSTRAR   = 5        # tope de filas en la figura (None = todas)
POSTPROCESO   = True     # limpieza morfológica + mayor componente conexa
RELLENAR      = True     # rellena los huecos negros dentro de la lesión
GUARDAR       = True     # guardar la figura en CARPETA_SALIDA

REDIMENSIONAR = True     # 256x256, como el resto del pipeline del TFG.
IMG_SIZE      = 256      # Ponlo a False para trabajar a resolución nativa

# --- validación cruzada ---
N_SPLITS      = 5        # K = 5 pliegues
VAL_FRACTION  = 0.20     # 20 % del 80 % restante = 16 % del total
RANDOM_STATE  = 42       # semilla fija -> pliegues y centroides reproducibles

# --- test fijo -------------------------------------------------------------
# Imágenes con imagen registrada del dataset: SIEMPRE son test, en todos los
# pliegues, y NUNCA entran en entrenamiento ni en validación.
INDICES_TEST_FIJOS = list(range(23, 28))   # 23, 24, 25, 26, 27

# --- K-means ---------------------------------------------------------------
MUESTRA_KMEANS = 20000   # nº de píxeles muestreados para ajustar los centroides.
                         # Después se asignan TODOS los píxeles al centroide más
                         # cercano: mismo resultado práctico y mucho más rápido.
KM_INTENTOS    = 3       # reinicios de K-means (se queda con la menor inercia)
KM_ITER_MAX    = 50
KM_EPS         = 1e-4
PESO_XY        = 0.5     # peso de las coordenadas (x, y) en los espacios "*_xy"

# Espacio de búsqueda de la configuración: (espacio, k, criterio).
#   espacio  -> características de cada píxel
#   k        -> nº de clústeres
#   criterio -> regla para decidir cuál de los k clústeres es la lesión
CONFIGS_CANDIDATAS = [
    ("lab_ab",    2, "rojizo"),    # crominancia: los hemangiomas son rojizos
    ("lab_ab",    3, "rojizo"),
    ("lab_ab",    4, "rojizo"),
    ("lab",       3, "rojizo"),    # incluye luminancia
    ("lab_ab_xy", 3, "rojizo"),    # color + posición -> clústeres más compactos
    ("lab_ab_xy", 4, "rojizo"),
    ("lab_ab",    3, "central"),   # el clúster más centrado en la imagen
    ("lab_ab_xy", 3, "central"),
    ("hsv",       3, "saturado"),  # el clúster más saturado
    ("rgb",       3, "oscuro"),    # el clúster más oscuro
    ("gris",      2, "oscuro"),    # equivalente aproximado a Otsu sobre gris
]

### 3. Importaciones

In [ ]:
import os
import h5py
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_fill_holes
from sklearn.model_selection import StratifiedKFold, train_test_split
from collections import Counter

cv2.setRNGSeed(RANDOM_STATE)   # K-means de OpenCV reproducible

### 4. Lectura del dataset

Se cargan **todas** las entradas (no una selección), porque la validación cruzada necesita el
conjunto completo.

In [ ]:
f  = h5py.File(RUTA_MAT, "r")
DS = f["DATASET_UNIDO"]

def leer_entrada(i):
    c = [f[r] for r in f[DS[i, 0]][()].ravel()]
    nombre = "".join(chr(x) for x in c[0][()].ravel())
    tipo   = "".join(chr(x) for x in c[2][()].ravel())
    mask   = (c[1][()].T > 0).astype(np.uint8)                          # (H, W) binaria
    img    = (np.transpose(c[3][()], (2, 1, 0)) * 255).clip(0, 255).astype(np.uint8)
    return nombre, tipo, img, mask


N = DS.shape[0]
nombres, tipos, imagenes, mascaras = [], [], [], []

for i in range(N):
    nombre, tipo, img, gt = leer_entrada(i)
    if REDIMENSIONAR:
        # área para la imagen, vecino más próximo para la máscara,
        # que así conserva su carácter estrictamente binario
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        gt  = cv2.resize(gt,  (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    nombres.append(nombre)
    tipos.append(tipo)
    imagenes.append(img)
    mascaras.append((gt > 0).astype(np.uint8))

etiquetas = np.array(tipos)
print(f"{N} imágenes cargadas. ")

130 imágenes cargadas. 


### 5. Segmentación por K-means

Tres piezas:

1. **Espacio de características** — cada píxel se convierte en un vector: color (LAB, RGB,
   HSV o gris) y, opcionalmente, sus coordenadas `(x, y)`. Todas las componentes se
   **tipifican** (media 0, desviación 1) y las coordenadas se ponderan con `PESO_XY`, lo
   que favorece clústeres espacialmente compactos sin que la posición domine al color.
2. **Agrupamiento** — `cv2.kmeans` sobre una muestra de píxeles (`MUESTRA_KMEANS`) para
   obtener los centroides; después se asigna **todo** el mapa al centroide más cercano.
3. **Selección del clúster de lesión** — K-means es no supervisado: numera los grupos
   arbitrariamente, así que hace falta una regla que diga cuál es la lesión. Los criterios
   disponibles son `rojizo` (mayor media del canal *a* de LAB), `oscuro` (menor media de
   gris), `saturado` (mayor media de S en HSV) y `central` (mayor concentración de píxeles
   hacia el centro de la imagen). Además se descartan los clústeres que ocupan más del
   90 % de la imagen, que casi siempre son piel o fondo.

**El criterio no usa el ground truth**, solo propiedades de la propia imagen; el ground
truth interviene únicamente al medir el Dice en la selección de configuración por pliegue.

In [ ]:
def caracteristicas(img, espacio):
    """Convierte la imagen en una matriz (H*W, D) de características por píxel.

    Cada componente se **tipifica** (media 0, desviación 1 dentro de la imagen) para
    que ninguna domine la distancia euclídea que usa K-means: sin tipificar, la
    crominancia varía poquísimo frente a las coordenadas y el agrupamiento acabaría
    siendo puramente espacial."""
    usar_xy = espacio.endswith("_xy")
    base    = espacio[:-3] if usar_xy else espacio

    suave = cv2.GaussianBlur(img, (5, 5), 0)
    H, W  = suave.shape[:2]

    if base == "gris":
        ch = cv2.cvtColor(suave, cv2.COLOR_RGB2GRAY)[:, :, None].astype(np.float32)
    elif base == "lab":
        ch = cv2.cvtColor(suave, cv2.COLOR_RGB2LAB).astype(np.float32)
    elif base == "lab_ab":
        ch = cv2.cvtColor(suave, cv2.COLOR_RGB2LAB)[:, :, 1:3].astype(np.float32)
    elif base == "rgb":
        ch = suave.astype(np.float32)
    elif base == "hsv":
        ch = cv2.cvtColor(suave, cv2.COLOR_RGB2HSV).astype(np.float32)
    else:
        raise ValueError(f"espacio desconocido: {espacio}")

    X = ch.reshape(-1, ch.shape[2])
    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-6)     # tipificación por canal

    if usar_xy:
        yy, xx = np.mgrid[0:H, 0:W]
        xy = np.stack([xx / (W - 1), yy / (H - 1)], axis=-1).reshape(-1, 2).astype(np.float32)
        xy = (xy - xy.mean(axis=0)) / (xy.std(axis=0) + 1e-6)
        X  = np.concatenate([X, PESO_XY * xy], axis=1)    # PESO_XY = importancia relativa

    return np.ascontiguousarray(X, dtype=np.float32), (H, W)


def mapa_criterio(img, criterio):
    """Mapa escalar (H, W) cuya media dentro de un clúster mide cómo de 'lesión' es.
    No usa el ground truth: solo propiedades de la imagen."""
    if criterio == "rojizo":
        return cv2.cvtColor(img, cv2.COLOR_RGB2LAB)[:, :, 1].astype(np.float32)
    if criterio == "oscuro":
        return -cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    if criterio == "saturado":
        return cv2.cvtColor(img, cv2.COLOR_RGB2HSV)[:, :, 1].astype(np.float32)
    if criterio == "central":
        H, W = img.shape[:2]
        yy, xx = np.mgrid[0:H, 0:W]
        d = np.sqrt(((xx - (W - 1) / 2) / (W / 2)) ** 2 +
                    ((yy - (H - 1) / 2) / (H / 2)) ** 2)
        return (1.0 - np.clip(d, 0, 1)).astype(np.float32)   # 1 en el centro, 0 en el borde
    raise ValueError(f"criterio desconocido: {criterio}")


def segmentar_kmeans(img, espacio="lab_ab", k=3, criterio="rojizo",
                     postproceso=True, rellenar=True):
    """K-means sobre los píxeles + selección del clúster que corresponde a la lesión."""
    X, (H, W) = caracteristicas(img, espacio)

    # Centroides ajustados sobre una muestra (rápido y prácticamente equivalente)
    rng = np.random.default_rng(RANDOM_STATE)
    if X.shape[0] > MUESTRA_KMEANS:
        muestra = X[rng.choice(X.shape[0], MUESTRA_KMEANS, replace=False)]
    else:
        muestra = X

    criterios_km = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
                    KM_ITER_MAX, KM_EPS)
    _, _, centros = cv2.kmeans(np.ascontiguousarray(muestra), k, None,
                               criterios_km, KM_INTENTOS, cv2.KMEANS_PP_CENTERS)

    # Todos los píxeles se asignan al centroide más cercano
    d   = ((X[:, None, :] - centros[None, :, :]) ** 2).sum(axis=2)
    lbl = d.argmin(axis=1).reshape(H, W)

    # ¿Cuál de los k clústeres es la lesión?
    mapa   = mapa_criterio(img, criterio)
    areas  = np.array([(lbl == j).mean() for j in range(k)])
    puntos = np.array([mapa[lbl == j].mean() if areas[j] > 0 else -np.inf
                       for j in range(k)])

    # Se descartan clústeres que ocupan casi toda la imagen (suelen ser el fondo/piel)
    validos = areas < 0.90
    if validos.any():
        puntos = np.where(validos, puntos, -np.inf)
    elegido = int(np.argmax(puntos))

    seg = (lbl == elegido).astype(np.uint8)

    if postproceso:
        kk = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        seg = cv2.morphologyEx(seg, cv2.MORPH_OPEN, kk)
        seg = cv2.morphologyEx(seg, cv2.MORPH_CLOSE, kk)
        n, etq = cv2.connectedComponents(seg)
        if n > 2:                       # conserva la mayor componente conexa
            areas_cc = [(etq == j).sum() for j in range(1, n)]
            seg = (etq == 1 + int(np.argmax(areas_cc))).astype(np.uint8)

    if rellenar:
        seg = binary_fill_holes(seg).astype(np.uint8)   # tapa huecos interiores

    return seg.astype(np.uint8)


# La misma imagen se segmenta muchas veces con la misma configuración (una por
# pliegue y por candidata), así que se memoriza el resultado. Es determinista y
# depende solo de (imagen, configuración): no introduce ninguna fuga de datos.
_CACHE_SEG = {}

def segmentar_cache(i, espacio, k, criterio):
    clave = (i, espacio, k, criterio, POSTPROCESO, RELLENAR)
    if clave not in _CACHE_SEG:
        _CACHE_SEG[clave] = segmentar_kmeans(imagenes[i], espacio, k, criterio,
                                             POSTPROCESO, RELLENAR)
    return _CACHE_SEG[clave]

### 6. Métricas y dibujo de contornos

In [ ]:
EPS = 1e-8

def matriz_confusion(pred, gt):
    """Recuento de VP, VN, FP y FN a nivel de píxel."""
    p, g = pred > 0, gt > 0
    return dict(vp=int(( p &  g).sum()), vn=int((~p & ~g).sum()),
                fp=int(( p & ~g).sum()), fn=int((~p &  g).sum()))

def suma_confusion(a, b):
    return {k: a[k] + b[k] for k in a}

def dice_de_confusion(c):
    return (2 * c["vp"] + EPS) / (2 * c["vp"] + c["fp"] + c["fn"] + EPS)

def metricas_de_confusion(c):
    return {
        "exactitud":     (c["vp"] + c["vn"]) / (c["vp"] + c["vn"] + c["fp"] + c["fn"] + EPS),
        "precision":     (c["vp"] + EPS) / (c["vp"] + c["fp"] + EPS),
        "sensibilidad":  (c["vp"] + EPS) / (c["vp"] + c["fn"] + EPS),
        "especificidad": (c["vn"] + EPS) / (c["vn"] + c["fp"] + EPS),
    }

def dice(a, b):
    return dice_de_confusion(matriz_confusion(a, b))


def evaluar(indices, espacio, k, criterio):
    """Evalúa una configuración de K-means sobre un subconjunto de índices.

    Distingue las dos formas de agregar, tal y como se explica en §6.2 de la memoria:
      * Dice -> se calcula por imagen y se promedia (macro). Cada lesión pesa igual.
      * F1   -> se acumulan VP, FP y FN de todas las imágenes y se calcula una sola
                vez sobre el total (micro). Cada píxel pesa igual, así que las
                lesiones grandes influyen más.
    Por eso Dice y F1 comparten fórmula pero no valor."""
    dices, total = [], dict(vp=0, vn=0, fp=0, fn=0)
    for i in indices:
        seg = segmentar_cache(i, espacio, k, criterio)
        c = matriz_confusion(seg, mascaras[i])
        dices.append(dice_de_confusion(c))
        total = suma_confusion(total, c)
    res = metricas_de_confusion(total)
    res["dice"] = float(np.mean(dices))     # macro, por imagen
    res["f1"]   = dice_de_confusion(total)  # micro, global
    return res


def elegir_configuracion(indices_ajuste):
    """Elige la configuración (espacio, k, criterio) que maximiza el Dice en
    entrenamiento + validación. Ninguna imagen de test interviene en esta decisión."""
    mejor, mejor_dice = None, -1.0
    for espacio, k, criterio in CONFIGS_CANDIDATAS:
        d = evaluar(indices_ajuste, espacio, k, criterio)["dice"]
        if d > mejor_dice:
            mejor, mejor_dice = (espacio, k, criterio), d
    return mejor, mejor_dice


def dibujar_contornos(img, seg, gt, grosor=None):
    o = img.copy()
    if grosor is None:
        grosor = max(2, int(round(max(img.shape[:2]) / 300)))   # grosor según tamaño
    cont_gt,  _ = cv2.findContours(gt.astype(np.uint8),  cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cont_seg, _ = cv2.findContours(seg.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(o, cont_gt,  -1, (255, 0, 0), grosor)   # ground truth -> rojo
    cv2.drawContours(o, cont_seg, -1, (0, 255, 0), grosor)   # K-means      -> verde
    return o

### 7. Validación cruzada estratificada de 5 pliegues con test fijo

Las imágenes de índice **23–27** (las que tienen imagen registrada) se reservan como
**test en todos los pliegues** y **nunca** participan en la elección de la configuración
de K-means (ni en entrenamiento ni en validación).

El resto del dataset se reparte con `StratifiedKFold`, de modo que en cada pliegue:

* **test** = pliegue rotatorio + las 5 imágenes fijas
* **train / val** = el 80 % restante de las imágenes rotatorias (80/20 interno)

Como las fijas se evalúan K veces (una por pliegue, con la configuración de cada uno),
su Dice fuera de pliegue se reporta como **media entre los K pliegues**.

In [ ]:
# --- conjunto de test fijo -------------------------------------------------
FIJOS = np.array(sorted(set(INDICES_TEST_FIJOS)), dtype=int)
assert FIJOS.min() >= 0 and FIJOS.max() < N, "INDICES_TEST_FIJOS fuera de rango"

# El resto del dataset es lo único que rota entre entrenamiento, validación y test
RESTO = np.array([i for i in range(N) if i not in set(FIJOS.tolist())], dtype=int)

print(f"Test fijo ({len(FIJOS)} imágenes, siempre en test, nunca en train/val):")
for i in FIJOS:
    print(f"   idx {i:>3}  {nombres[i]}")
print(f"Imágenes que rotan en la validación cruzada: {len(RESTO)}\n")

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

resultados_pliegue       = []   # test completo = pliegue rotatorio + fijas
resultados_pliegue_rot   = []   # solo la parte rotatoria del test
resultados_pliegue_fijos = []   # solo las imágenes fijas
configs_elegidas         = []

# predicción fuera de pliegue para cada imagen
seg_oof    = [None] * N
dice_oof   = np.zeros(N)
pliegue_de = np.zeros(N, dtype=int)
dice_fijos_por_pliegue = {i: [] for i in FIJOS}   # las fijas se predicen K veces

print(f"Validación cruzada estratificada de {N_SPLITS} pliegues — K-means\n")

for pliegue, (pos_trval, pos_test) in enumerate(
        skf.split(np.zeros(len(RESTO)), etiquetas[RESTO]), 1):

    idx_trval    = RESTO[pos_trval]
    idx_test_rot = RESTO[pos_test]
    # El test de cada pliegue = parte rotatoria + las imágenes fijas
    idx_test     = np.concatenate([idx_test_rot, FIJOS])

    # Separación interna entrenamiento / validación
    try:
        idx_train, idx_val = train_test_split(
            idx_trval, test_size=VAL_FRACTION,
            stratify=etiquetas[idx_trval], random_state=RANDOM_STATE)
    except ValueError:
        # alguna clase tiene muy pocas muestras para estratificar el subreparto
        idx_train, idx_val = train_test_split(
            idx_trval, test_size=VAL_FRACTION, random_state=RANDOM_STATE)

    # La configuración se decide sobre entrenamiento + validación...
    (espacio, k, criterio), dice_ajuste = elegir_configuracion(
        np.concatenate([idx_train, idx_val]))
    # ...y se aplica al pliegue de test, no visto en la selección.
    res       = evaluar(idx_test,     espacio, k, criterio)
    res_rot   = evaluar(idx_test_rot, espacio, k, criterio)
    res_fijos = evaluar(FIJOS,        espacio, k, criterio)

    for i in idx_test:
        s = segmentar_cache(i, espacio, k, criterio)
        d = dice(s, mascaras[i])
        seg_oof[i]    = s          # para las fijas queda la del último pliegue
        dice_oof[i]   = d
        pliegue_de[i] = pliegue
        if i in dice_fijos_por_pliegue:
            dice_fijos_por_pliegue[i].append(d)

    resultados_pliegue.append(res)
    resultados_pliegue_rot.append(res_rot)
    resultados_pliegue_fijos.append(res_fijos)
    configs_elegidas.append((espacio, k, criterio))

# El Dice fuera de pliegue de las imágenes fijas se promedia entre los K pliegues
for i in FIJOS:
    dice_oof[i] = float(np.mean(dice_fijos_por_pliegue[i]))

Test fijo (5 imágenes, siempre en test, nunca en train/val):
   idx  23  imagen20.jpg
   idx  24  imagen21.jpg
   idx  25  imagen24.jpg
   idx  26  imagen29.jpg
   idx  27  imagen3.jpg
Imágenes que rotan en la validación cruzada: 125

Validación cruzada estratificada de 5 pliegues — K-means



### 8. Resumen

In [ ]:
CLAVES  = ["exactitud", "sensibilidad", "especificidad", "precision", "dice", "f1"]
NOMBRES = {"exactitud": "Exactitud", "precision": "Precisión",
           "sensibilidad": "Sensibilidad", "especificidad": "Especificidad",
           "dice": "Dice", "f1": "F1"}

def resumir(lista):
    return {m: (float(np.mean([r[m] for r in lista])),
                float(np.std ([r[m] for r in lista]))) for m in CLAVES}

resumen       = resumir(resultados_pliegue)
resumen_rot   = resumir(resultados_pliegue_rot)
resumen_fijos = resumir(resultados_pliegue_fijos)

print("=" * 72)
print(f"K-means — validación cruzada de {N_SPLITS} pliegues ")
print("=" * 72)
print(f"{'Métrica':<16}{'Test total':>14}")
print("-" * 72)
for m in CLAVES:
    print(f"{NOMBRES[m]:<16}{resumen[m][0]:>14.4f}")
print("-" * 72)





K-means — validación cruzada de 5 pliegues 
Métrica             Test total
------------------------------------------------------------------------
Exactitud               0.8623
Sensibilidad            0.8868
Especificidad           0.9003
Precisión               0.8762
Dice                    0.8180
F1                      0.7821
------------------------------------------------------------------------
